In [1]:
# ## install finrl library
!pip install wrds
!pip install swig
!pip install -q condacolab
import condacolab
condacolab.install()
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.9 MB/s eta 0:00:00
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:20
🔁 Restarting kernel...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libgl1-mesa-glx:amd64.
(Reading database ... 124926 files and directories currently installed.)
Preparing to unpack .../libgl1-mesa-glx_23.0.4-0ubuntu1~22.04.1_amd64.deb ...
Unpacking libgl1-mesa-glx:amd64 (23.0.4-0ubuntu1~22.04.1) ...
Selecting previously unselected package swig4.0.
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.

In [1]:
import warnings
warnings.filterwarnings("ignore")


In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent,DRLEnsembleAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

from pprint import pprint

import sys
sys.path.append("../FinRL-Library")

import itertools

/usr/local/lib/python3.11/site-packages/pandas_datareader/compat/__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)


In [3]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

In [ ]:
#Saved the preprocessed historical_data.csv as data.csv

df = pd.read_csv('data.csv')

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
#Selecting the indicators 

INDICATORS = ['macd',
               'rsi_30',
               'cci_30',
               'dx_30']

In [6]:
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df)

Successfully added technical indicators
Successfully added turbulence index


In [7]:
df.head()

,date,close,high,low,open,volume,tic,day,daily_return
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3,NaN
1,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3,12.123810
2,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3,-0.782293
3,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4,0.632084
4,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4,44.075824


In [9]:
len(df)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


4066965

In [10]:
stock_dimension = len(processed.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 3, State Space: 19


In [ ]:
env_kwargs = {
    "hmax": 100, #maximum number of shares
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001, #percentage transaction cost
    "sell_cost_pct": 0.001, #percentage transaction cost
    "state_space": state_space, #umber of features in the state representation
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension, #number of actions available
    "reward_scaling": 1e-4, #scaling factor for rewards
    "print_verbosity":5

}


In [12]:
df.head()

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,date,close,high,low,open,volume,tic,day,daily_return
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3,NaN
1,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3,12.123810
2,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3,-0.782293
3,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4,0.632084
4,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4,44.075824


In [15]:
df.date.value_counts()

,count
date,
2024-12-30,1621
2024-12-02,1620
2024-12-13,1619
2024-12-20,1618
2024-12-05,1618
...,...
2004-09-06,3
2004-10-11,3
2004-12-27,3


In [16]:
df.date.info()

<class 'pandas.core.series.Series'>
RangeIndex: 4066965 entries, 0 to 4066964
Series name: date
Non-Null Count    Dtype 
--------------    ----- 
4066965 non-null  object
dtypes: object(1)
memory usage: 31.0+ MB


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [19]:
(len(df) * 0.8)

3253572.0

In [18]:
df = df.sort_values(by='date').reset_index(drop=True)

In [22]:
split_idx = int(len(df) * 0.8)

In [23]:
# Extract training and validation periods directly
TRAIN_START_DATE = df.iloc[0]['date']
TRAIN_END_DATE = df.iloc[split_idx - 1]['date']

TEST_START_DATE = df.iloc[split_idx]['date']
TEST_END_DATE = df.iloc[-1]['date']

In [24]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df=processed,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window,
                 validation_window=validation_window,
                 **env_kwargs)

In [25]:
SAC_model_kwargs = {
    "batch_size": 64,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}